In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기
# ============================================================

train = pd.read_parquet(PROJECT_ROOT / 'data' / '3_train.parquet')
val = pd.read_parquet(PROJECT_ROOT / 'data' / '3_val.parquet')

# from sklearn import set_config
# set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

In [2]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

# 종속변수군
target_col = 'target_maxrise_6h'

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]

# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = [
    col for col in (train.select_dtypes(include='number').columns)
    if col not in ['event_id', target_col]]  # 'station_id', 'timestamp'는 미포함
origin_cols = wl_cols + rain_cols + env_cols

assert len(num_cols) == 31, "'num_cols' 개수 이상"  # 종속변수 제외됨
assert (set(train.columns) - set(num_cols)) == {'station_id', 'timestamp', 'event_id', target_col}, "'num_cols' 확인필요"
assert set(origin_cols).issubset(num_cols), "'origin_cols' 확인필요"

In [3]:
# ============================================================
# 2. 종속변수 분리
# ============================================================

y_train = train[target_col]
y_val = val[target_col]

In [12]:
# ============================================================
# 3. 최종 독립변수 정의(OHE 포함)
# ============================================================

# 기존 독립변수 정의
cols = origin_cols + wl_diff_cols

# 독립변수에 OHE 추가하기
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    handle_unknown='ignore',  # train에 없던 범주가 나와도 에러 안띄움
    sparse_output=True  # 희소행렬로 반환(원핫인코딩을 가볍게 만들어줌, 빅데이터에서 필수)
)

ohe.fit(train[['station_id']])  # skleran 인코더 2차원 행렬 필수

assert set(val['station_id']).issubset(set(train['station_id'])), "val에 변환되지 않은 'station_id' 발생 가능"

train_ohe = ohe.transform(train[['station_id']])
val_ohe = ohe.transform(val[['station_id']])
# 여기까지는 변환, 실제 데이터에 합친건 아님

# 합치기
from scipy.sparse import hstack

X_train = hstack([
    train[cols],
    train_ohe
]).tocsr()

X_val = hstack([
    val[cols],
    val_ohe
]).tocsr()

# 행 개수 검증
assert X_train.shape[0] == len(train), "Train 행 개수 불일치"
assert X_val.shape[0] == len(val), "Val 행 개수 불일치"
# 열 개수 검증
assert X_train.shape[1] == len(cols) + train_ohe.shape[1], "Train 열 개수 불일치"
assert X_val.shape[1] == X_train.shape[1], "Train과 Val 열 개수 불일치"

In [14]:
# ============================================================
# 4. Cat 최적화-1: best iteration 탐색
# ============================================================
# 사유: 모델비교에서 트리개수가 기본값의 최대치인 1000으로 확인, 학습이 부족했을 가능성
# iteration=3000, early_stopping_rounds=100, eval_set=(X_val, y_val) 적용 전
# best_iteration이 999(1000번째)인지 확인 필요

# CatBoost 기본모델: iteration=1000으로 best_iteratoin 확인

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

results = []

# 모델링
model1 = CatBoostRegressor(
    verbose=500,
    random_state=42
)

model1.fit(
    X_train, y_train,
    early_stopping_rounds=100,
    eval_set=(X_val, y_val)
)

pred_val = model1.predict(X_val)
pred_train = model1.predict(X_train)

rmse_val = root_mean_squared_error(y_val, pred_val)
rmse_train = root_mean_squared_error(y_train, pred_train)

gap = rmse_val - rmse_train

results.append({
    'Model': 'Cat_iter_1000',
    'best_iteration': model1.get_best_iteration(),
    'val_RMSE': rmse_val,
    'train_RMSE': rmse_train,
    'Gap': gap
    })

print(results[-1])


Learning rate set to 0.139602
0:	learn: 0.3771988	test: 0.3324216	best: 0.3324216 (0)	total: 59.6ms	remaining: 59.6s
500:	learn: 0.2053794	test: 0.1969128	best: 0.1969128 (500)	total: 23.2s	remaining: 23.1s
999:	learn: 0.1901659	test: 0.1950727	best: 0.1950234 (990)	total: 45.5s	remaining: 0us

bestTest = 0.1950233664
bestIteration = 990

Shrink model to first 991 iterations.
{'Model': 'Cat_iter_1000', 'best_iteration': 990, 'val_RMSE': 0.19502336643352713, 'train_RMSE': 0.19040694727729632, 'Gap': 0.0046164191562308166}


In [ ]:
# ============================================================
# 5. Cat 최적화-2: iteration: 3000으로 상향
# ============================================================
# 사유: 모델비교에서 트리개수가 기본값의 최대치인 1000으로 확인, 학습이 부족했을 가능성
# iteration=3000, early_stopping_rounds=100, eval_set=(X_val, y_val) 적용

# CatBoost, iteration=3000으로 best_iteratoin 확인

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

# 모델링
model2 = CatBoostRegressor(
    iterations=3000,
    verbose=500,
    random_state=42
)

model2.fit(
    X_train, y_train,
    early_stopping_rounds=100,
    eval_set=(X_val, y_val)
)

pred_val = model2.predict(X_val)
pred_train = model2.predict(X_train)

rmse_val = root_mean_squared_error(y_val, pred_val)
rmse_train = root_mean_squared_error(y_train, pred_train)

gap = rmse_val - rmse_train

results.append({
    'Model': 'Cat_iter_3000',
    'best_iteration': model2.get_best_iteration(),
    'val_RMSE': rmse_val,
    'train_RMSE': rmse_train,
    'Gap': gap
    })

print(results[-1])

Learning rate set to 0.071425
0:	learn: 0.3895086	test: 0.3444559	best: 0.3444559 (0)	total: 55.9ms	remaining: 2m 47s
500:	learn: 0.2173392	test: 0.1993331	best: 0.1993276 (498)	total: 28.6s	remaining: 2m 22s
1000:	learn: 0.2044676	test: 0.1962161	best: 0.1962138 (997)	total: 58.7s	remaining: 1m 57s
1500:	learn: 0.1959571	test: 0.1950167	best: 0.1950026 (1478)	total: 1m 30s	remaining: 1m 30s
2000:	learn: 0.1892894	test: 0.1942970	best: 0.1942691 (1982)	total: 2m 1s	remaining: 1m
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.194134478
bestIteration = 2337

Shrink model to first 2338 iterations.
{'Model': 'Cat_iter_3000', 'best_iteration': 2337, 'val_RMSE': 0.1941344779529384, 'train_RMSE': 0.18545961081761345, 'Gap': 0.008674867135324948}


In [ ]:
# ============================================================
# 6. Cat 최적화-3: OHE 대신 cat모델 자체 인코더 사용
# ============================================================

# 독립변수 정의
cols2 = origin_cols + wl_diff_cols + ['station_id']
X_train = train[cols2].copy()  # 'station_id' 문자열 변환 예정이므로 copy() 필요
X_val = val[cols2].copy()

X_train['station_id'] = X_train['station_id'].astype(str)
X_val['station_id'] = X_val['station_id'].astype(str)

# 모델링
model3 = CatBoostRegressor(
    iterations=3000,
    verbose=500,
    random_state=42
)

model3.fit(
    X_train, y_train,
    cat_features=['station_id'],  # 자체 인코더 설정 필수
    early_stopping_rounds=100,
    eval_set=(X_val, y_val)
)

pred_val = model3.predict(X_val)
pred_train = model3.predict(X_train)

rmse_val = root_mean_squared_error(y_val, pred_val)
rmse_train = root_mean_squared_error(y_train, pred_train)

gap = rmse_val - rmse_train

results.append({
    'Model': 'Cat_no_ohe',
    'best_iteration': model3.get_best_iteration(),
    'val_RMSE': rmse_val,
    'train_RMSE': rmse_train,
    'Gap': gap
    })

print(results[-1])

Learning rate set to 0.071425
0:	learn: 0.3895443	test: 0.3446945	best: 0.3446945 (0)	total: 278ms	remaining: 13m 53s
500:	learn: 0.2155696	test: 0.1961997	best: 0.1961997 (500)	total: 1m 28s	remaining: 7m 20s
1000:	learn: 0.2029600	test: 0.1931596	best: 0.1931596 (1000)	total: 2m 54s	remaining: 5m 48s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.1924565473
bestIteration = 1328

Shrink model to first 1329 iterations.
{'Model': 'Cat_no_ohe', 'best_iteration': 1328, 'val_RMSE': 0.1924565473356066, 'train_RMSE': 0.19649307093895113, 'Gap': -0.004036523603344527}


### 최종 모델
**iteration=3000**

**CatBoost 자체 인코더 사용**

In [4]:
# ============================================================
# 7. Cat 최적화-3: K-Fold 교차검증
# ============================================================

# 각각 예측데이터 저장해서 OOF도 만들고 플롯도 해야함

# ------------------------------------------------------------
# 3-1. 기존 Fold 매칭 검증 및 행별 Fold 번호 매핑용 정보
# ------------------------------------------------------------

kf_groups = train['event_id']

assert not kf_groups.isna().any()

print('<시간사상 Group 확인>')
print(f'전체 학습 행 수 : {len(train):,}')
print(f'시간사상 개수   : {kf_groups.nunique():,}')
print()

# ------------------------------------------------------------
# 3-2. 기존 OOF에서 추출한 사상별 fold 배정 불러오기
# ------------------------------------------------------------

N_SPLITS = 5

kf_fold_map = pd.read_parquet(
    PROJECT_ROOT / 'data' / '0_kf_train_match.parquet'
)

# 검증
assert kf_fold_map.columns.tolist() == ['event_id', 'fold_num']
assert kf_fold_map['event_id'].is_unique
assert set(kf_fold_map['fold_num']) == set(range(1, N_SPLITS + 1))  # 1 ~ N_SPLITS 까지
assert not kf_fold_map.isna().any().any()  # 전체에 결측치 없어야 함
assert set(kf_groups) == set(kf_fold_map['event_id'])  # 실제 'event_id'와 매핑규칙이 동일해야함

# 시리즈1.map(시리즈2): 시리즈1을 시리즈2(매핑규칙)의 값으로 치환, 시리즈2의 인덱스가 시리즈1과 같은 변수여야 함
kf_fold_num = kf_groups.map(  # kf_groups는 event_id 시리즈, 치환대상
    kf_fold_map.set_index('event_id')['fold_num']  # 매핑규칙: evnet_id을 'fold_num'으로 치환
    )
assert kf_fold_num.notna().all()

import numpy as np
# 행 위치 배열만 보관한다. 데이터셋은 복사하지 않는다.
# 학습·진단·OOF fold 기록에서 모두 같은 분할을 재사용한다.
# np.flatnonzero(조건문): 조건을 만족하는 값의 숫자인덱스를 반환하는 함수
kf_splits = []

for fold in range(1, N_SPLITS + 1):

    # np.flatnonzero(조건문): 조건에 맞는 숫자인덱스(순서인덱스)를 반환
    # flat: 1차원으로 펼쳐서, nonzero: 0(False)이 아닌(조건에 맞는) -> 위치를 반환
    train_idx = np.flatnonzero(kf_fold_num != fold)
    val_idx = np.flatnonzero(kf_fold_num == fold)

    kf_splits.append(
        (train_idx, val_idx)
    )
    # 결과물: Fold별 (train_idx, val_idx)가 포함된 리스트

# ------------------------------------------------------------
# 3-3. Fold 검증
# ------------------------------------------------------------

for fold, (train_idx, valid_idx) in enumerate(  # 각각의 Fold별 (번호, (train_idx, val_idx)) 형태를 반환
    kf_splits,
    start=1  # 숫자의 시작은 1부터(0이 아니라)
):
    # 고정한 분할의 사상 중복 여부 검증
    train_groups = set(kf_groups.iloc[train_idx])  # train_idx에 해당하는 'event_id'만 반환
    val_groups = set(kf_groups.iloc[val_idx])

    overlap = train_groups & val_groups  # 교집합은 0이어야함

    print(
        f'Fold {fold} | '
        f'train={len(train_idx):,} | '
        f'valid={len(valid_idx):,} | '
        f'train events={len(train_groups):,} | '
        f'valid events={len(val_groups):,} | '
        f'event overlap={len(overlap)}'
    )

<시간사상 Group 확인>
전체 학습 행 수 : 616,293
시간사상 개수   : 98

Fold 1 | train=493,297 | valid=122,996 | train events=82 | valid events=20 | event overlap=20
Fold 2 | train=492,894 | valid=123,399 | train events=77 | valid events=20 | event overlap=20
Fold 3 | train=492,894 | valid=123,399 | train events=77 | valid events=20 | event overlap=20
Fold 4 | train=493,077 | valid=123,216 | train events=78 | valid events=20 | event overlap=20
Fold 5 | train=493,010 | valid=123,283 | train events=78 | valid events=20 | event overlap=0


In [5]:
# ------------------------------------------------------------
# 3-4. K-Fold 교차검증: OOF 데이터셋, RMSE 산정
# ------------------------------------------------------------

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

# 각각 예측데이터 저장해서 OOF도 만들고 플롯도 해야함

# 메모리 사용을 줄이고, 효율적 코딩을 위해서는 폴드별로 데이터셋을 저장 X
# 데이터 분할 + 모델링을 한 세트로 for문을 구현

# 독립변수 정의
cols2 = origin_cols + wl_diff_cols + ['station_id']
X_train = train[cols2].copy()  # 'station_id' 문자열 변환 예정이므로 copy() 필요
X_val = val[cols2].copy()

X_train['station_id'] = X_train['station_id'].astype(str)
X_val['station_id'] = X_val['station_id'].astype(str)

# K-Fold 설계
kf_results = []  # Fold별 평가점수 등 담을 리스트
pred_kf_oof = np.full(len(X_train), np.nan)  # 빈 배열 만들기

for fold_num, (train_idx, val_idx) in enumerate(
    kf_splits,
    start=1
):
    # fold별 데이터 분할
    X_train_kf = X_train.iloc[train_idx]
    X_val_kf = X_train.iloc[val_idx]

    y_train_kf = y_train.iloc[train_idx]
    y_val_kf = y_train.iloc[val_idx]

    # 모델 정의
    model_cat_kf = CatBoostRegressor(
        random_state=42,
        iterations=3000,
        cat_features=['station_id']  # CatBoost 전용 범주형변수 매개변수
    )

    # 학습
    model_cat_kf.fit(
        X_train_kf,
        y_train_kf,
        early_stopping_rounds=100,
        eval_set=(X_val_kf, y_val_kf),
        verbose=500
    )

    # 모델 저장: Val 예측용(RMSE 비교를 위해)
    model_cat_kf.save_model(
        PROJECT_ROOT / 'model' / 'cat_kf_train' / f'cat_train_fold_{fold_num}.cbm'
    )

    # 예측
    pred_kf_va = model_cat_kf.predict(X_val_kf)
    pred_kf_tr = model_cat_kf.predict(X_train_kf) # train_RMSE 구하는 용도
    pred_kf_oof[val_idx] = pred_kf_va  # 해당 Fold의 예측값 저장

    # 평가
    rmse_val = root_mean_squared_error(y_val_kf, pred_kf_va)
    rmse_train = root_mean_squared_error(y_train_kf, pred_kf_tr)
    gap = rmse_val - rmse_train

    # 평가 결과 저장
    kf_results.append({
        'fold': fold_num,
        'val_rmse': rmse_val,
        'train_rmse': rmse_train,
        'Gap': gap,
        'best_iteration': model_cat_kf.best_iteration_
    })

    # 결과 출력
    print(
        f'Fold {fold_num} | '
        f'Val={rmse_val:.4f} | '
        f'Train={rmse_train:.4f} | '
        f'Gap={gap:.4f} | '
        f'Best iter={model_cat_kf.best_iteration_}'
    )


# 예측값 누락 확인
assert len(pred_kf_oof) == len(train), 'pred_kf_oof 행 개수 이상'
assert not np.isnan(pred_kf_oof).any(), 'pred_kf_oof 결측치 발생'

# 전체 OOF RMSE
oof_rmse = root_mean_squared_error(y_train, pred_kf_oof)

print(f'OOF RMSE: {oof_rmse:.4f}')

# 전체 OOF 저장
# 합칠때 넘파이 배열은 순서대로 그대로 합쳐지고, 시리즈는 인덱스에 따라 재배열됨
# pred_kf_oof[val_idx] = pred_kf_va 이렇게 val_idx(행위치)로 배열했으므로 모두 넘파이로 그대로 붙이는 것이 안전!
df_oof = pd.DataFrame({
    'event_id': train['event_id'].to_numpy(),
    'y_train': y_train.to_numpy(),
    'y_pred': pred_kf_oof,
    'fold': kf_fold_num.to_numpy()  # train['event_id']를 'fold_num'으로 치환한 시리즈
}, index=train.index)

df_oof.to_parquet( PROJECT_ROOT / 'data' / '0_kf_oof.parquet')

Learning rate set to 0.068972
0:	learn: 0.3781036	test: 0.4349251	best: 0.4349251 (0)	total: 399ms	remaining: 19m 55s
500:	learn: 0.2029928	test: 0.2737709	best: 0.2737709 (500)	total: 1m 27s	remaining: 7m 15s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.27282786
bestIteration = 723

Shrink model to first 724 iterations.
Fold 1 | Val=0.2728 | Train=0.1951 | Gap=0.0777 | Best iter=723
Learning rate set to 0.068963
0:	learn: 0.4063962	test: 0.3167504	best: 0.3167504 (0)	total: 178ms	remaining: 8m 53s
500:	learn: 0.2286019	test: 0.1511527	best: 0.1511527 (500)	total: 1m 11s	remaining: 5m 56s
1000:	learn: 0.2149104	test: 0.1487667	best: 0.1487667 (1000)	total: 2m 15s	remaining: 4m 29s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.148157383
bestIteration = 1293

Shrink model to first 1294 iterations.
Fold 2 | Val=0.1482 | Train=0.2083 | Gap=-0.0601 | Best iter=1293
Learning rate set to 0.068963
0:	learn: 0.3912895	test: 0.3855159	best: 0.385515

In [14]:
# ------------------------------------------------------------
# 3-5. K-Fold 교차검증: 최종 Val 검증
# ------------------------------------------------------------

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

# Fold별 예측값 저장
preds = []

for fold_num in range(1, 6):

    # 모델 불러오기
    model = CatBoostRegressor()

    model.load_model(
        str(PROJECT_ROOT / 'model' / 'cat_kf_train' / f'cat_train_fold_{fold_num}.cbm')
    )

    # Validation 예측
    pred = model.predict(X_val)

    # 예측값 저장
    preds.append(pred)

# 5개 모델의 예측값 평균
df_preds = pd.DataFrame(preds).T
df_preds.columns = [f'model{i}' for i in range(1, 6)]
df_preds['mean_pred'] = df_preds.mean(axis=1)
display(df_preds.head())

rmse_kf_ensemble = root_mean_squared_error(y_val, df_preds['mean_pred'])
print('K-Fold 교차검증 모델 앙상블(평균) Val RMSE: ', rmse_kf_ensemble)

,model1,model2,model3,model4,model5,mean_pred
0,0.017129,0.023891,0.024285,0.016363,0.018744,0.020082
1,-0.002336,-0.001113,-0.003460,-0.001681,-0.000984,-0.001915
2,-0.000980,-0.005865,0.003184,-0.004726,-0.003957,-0.002469
3,0.001501,-0.004053,-0.010057,-0.002557,-0.005670,-0.004167
4,0.018934,0.020373,0.015442,0.016391,0.010435,0.016315


K-Fold 교차검증 모델 앙상블(평균) Val RMSE:  0.19142102902849126


In [15]:
# ============================================================
# 8. Cat 최적화-4: K-Fold 교차검증(with rain_int_cols)
# ============================================================

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

# 각각 예측데이터 저장해서 OOF도 만들고 플롯도 해야함

# 메모리 사용을 줄이고, 효율적 코딩을 위해서는 폴드별로 데이터셋을 저장 X
# 데이터 분할 + 모델링을 한 세트로 for문을 구현

# 독립변수 정의
cols3 = origin_cols + wl_diff_cols + rain_int_cols + ['station_id']
X_train = train[cols3].copy()  # 'station_id' 문자열 변환 예정이므로 copy() 필요
X_val = val[cols3].copy()

X_train['station_id'] = X_train['station_id'].astype(str)
X_val['station_id'] = X_val['station_id'].astype(str)

# K-Fold 설계
kf_results_2 = []  # Fold별 평가점수 등 담을 리스트
pred_kf_oof_2 = np.full(len(X_train), np.nan)  # 빈 배열 만들기

for fold_num, (train_idx, val_idx) in enumerate(
    kf_splits,
    start=1
):
    # fold별 데이터 분할
    X_train_kf = X_train.iloc[train_idx]
    X_val_kf = X_train.iloc[val_idx]

    y_train_kf = y_train.iloc[train_idx]
    y_val_kf = y_train.iloc[val_idx]

    # 모델 정의
    model_cat_kf = CatBoostRegressor(
        random_state=42,
        iterations=3000,
        cat_features=['station_id']  # CatBoost 전용 범주형변수 매개변수
    )

    # 학습
    model_cat_kf.fit(
        X_train_kf,
        y_train_kf,
        early_stopping_rounds=100,
        eval_set=(X_val_kf, y_val_kf),
        verbose=500
    )

    # 모델 저장: Val 예측용(RMSE 비교를 위해)
    model_cat_kf.save_model(
        PROJECT_ROOT / 'model' / 'cat_kf_train_rain_int_cols' / f'cat_train_rain_int_fold_{fold_num}.cbm'
    )

    # 예측
    pred_kf_va = model_cat_kf.predict(X_val_kf)
    pred_kf_tr = model_cat_kf.predict(X_train_kf) # train_RMSE 구하는 용도
    pred_kf_oof_2[val_idx] = pred_kf_va  # 해당 Fold의 예측값 저장

    # 평가
    rmse_val = root_mean_squared_error(y_val_kf, pred_kf_va)
    rmse_train = root_mean_squared_error(y_train_kf, pred_kf_tr)
    gap = rmse_val - rmse_train

    # 평가 결과 저장
    kf_results_2.append({
        'fold': fold_num,
        'val_rmse': rmse_val,
        'train_rmse': rmse_train,
        'Gap': gap,
        'best_iteration': model_cat_kf.best_iteration_
    })

    # 결과 출력
    print(
        f'Fold {fold_num} | '
        f'Val={rmse_val:.4f} | '
        f'Train={rmse_train:.4f} | '
        f'Gap={gap:.4f} | '
        f'Best iter={model_cat_kf.best_iteration_}'
    )

# 예측값 누락 확인
assert len(pred_kf_oof_2) == len(train), 'pred_kf_oof 행 개수 이상'
assert not np.isnan(pred_kf_oof_2).any(), 'pred_kf_oof 결측치 발생'

# 전체 OOF RMSE
oof_rmse_2 = root_mean_squared_error(y_train, pred_kf_oof_2)

print(f'OOF RMSE: {oof_rmse_2:.4f}')

# 전체 OOF 저장
# 합칠때 넘파이 배열은 순서대로 그대로 합쳐지고, 시리즈는 인덱스에 따라 재배열됨
# pred_kf_oof[val_idx] = pred_kf_va 이렇게 val_idx(행위치)로 배열했으므로 모두 넘파이로 그대로 붙이는 것이 안전!
df_oof_2 = pd.DataFrame({
    'event_id': train['event_id'].to_numpy(),
    'y_train': y_train.to_numpy(),
    'y_pred': pred_kf_oof_2,
    'fold': kf_fold_num.to_numpy()  # train['event_id']를 'fold_num'으로 치환한 시리즈
}, index=train.index)

df_oof_2.to_parquet( PROJECT_ROOT / 'data' / '0_kf_oof_rain_int_cols.parquet')

Learning rate set to 0.068972
0:	learn: 0.3781888	test: 0.4351678	best: 0.4351678 (0)	total: 163ms	remaining: 8m 8s
500:	learn: 0.2018008	test: 0.2730497	best: 0.2730326 (490)	total: 1m 18s	remaining: 6m 29s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.2726256368
bestIteration = 578

Shrink model to first 579 iterations.
Fold 1 | Val=0.2726 | Train=0.1978 | Gap=0.0748 | Best iter=578
Learning rate set to 0.068963
0:	learn: 0.4063712	test: 0.3168128	best: 0.3168128 (0)	total: 158ms	remaining: 7m 52s
500:	learn: 0.2276720	test: 0.1502108	best: 0.1502107 (499)	total: 1m 17s	remaining: 6m 28s
1000:	learn: 0.2132281	test: 0.1478458	best: 0.1478458 (1000)	total: 2m 21s	remaining: 4m 43s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.1473218804
bestIteration = 1266

Shrink model to first 1267 iterations.
Fold 2 | Val=0.1473 | Train=0.2064 | Gap=-0.0590 | Best iter=1266
Learning rate set to 0.068963
0:	learn: 0.3912692	test: 0.3853778	best: 0.38537

In [16]:
# ------------------------------------------------------------
# 8-2. K-Fold 교차검증: 최종 Val 검증(with rain_int_cols)
# ------------------------------------------------------------

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

# Fold별 예측값 저장
preds_rain_int = []

for fold_num in range(1, 6):

    # 모델 불러오기
    model = CatBoostRegressor()

    model.load_model(
        str(PROJECT_ROOT / 'model' / 'cat_kf_train_rain_int_cols' / f'cat_train_rain_int_fold_{fold_num}.cbm')
    )

    # Validation 예측
    pred = model.predict(X_val)

    # 예측값 저장
    preds_rain_int.append(pred)

# 5개 모델의 예측값 평균
df_preds_rain_int = pd.DataFrame(preds_rain_int).T
df_preds_rain_int.columns = [f'model{i}' for i in range(1, 6)]
df_preds_rain_int['mean_pred'] = df_preds_rain_int.mean(axis=1)
display(df_preds_rain_int.head())

rmse_kf_ensemble_rain_int = root_mean_squared_error(y_val, df_preds_rain_int['mean_pred'])
print('K-Fold 교차검증(rain_int_cols 포함) 모델 앙상블(평균) Val RMSE: ', rmse_kf_ensemble_rain_int)

,model1,model2,model3,model4,model5,mean_pred
0,0.024878,0.021302,0.021387,0.010722,0.013934,0.018445
1,-0.001442,-0.001490,-0.006721,-0.001107,-0.003871,-0.002926
2,0.000997,-0.005951,-0.001641,-0.001950,0.000963,-0.001516
3,0.000847,-0.003823,-0.013647,-0.005122,0.000893,-0.004171
4,0.010919,0.006084,0.018166,0.022683,0.010443,0.013659


K-Fold 교차검증(rain_int_cols 포함) 모델 앙상블(평균) Val RMSE:  0.19223501035081386


In [19]:
# ============================================================
# 9. 결과정리
# ============================================================
# rain_int_cols 미포함이 OOF RMSE가 더 우수함, 채택

df_kf_results = pd.DataFrame(kf_results)
df_kf_results_2 = pd.DataFrame(kf_results_2)
print('========== K-Fold 교차검증(without rain_int_cols) ==========')
print(df_kf_results)
print(f'OOF RMSE: {oof_rmse:.4f}')
print('K-Fold 교차검증 모델 앙상블(평균) Val RMSE: ', round(rmse_kf_ensemble,4))
print()
print('========== K-Fold 교차검증(rain_int_cols) ==========')
print(df_kf_results_2)
print(f'OOF RMSE: {oof_rmse_2:.4f}')
print('K-Fold 교차검증(rain_int_cols 포함) 모델 앙상블(평균) Val RMSE: ', round(rmse_kf_ensemble_rain_int,4))

========== K-Fold 교차검증(without rain_int_cols) ==========
   fold  val_rmse  train_rmse       Gap  best_iteration
0     1  0.272828    0.195080  0.077748             723
1     2  0.148157    0.208262 -0.060104            1293
2     3  0.227583    0.180179  0.047404            2608
3     4  0.271230    0.187757  0.083473             982
4     5  0.218099    0.189729  0.028370            1720
OOF RMSE: 0.2320
K-Fold 교차검증 모델 앙상블(평균) Val RMSE:  0.1914

========== K-Fold 교차검증(rain_int_cols) ==========
   fold  val_rmse  train_rmse       Gap  best_iteration
0     1  0.272626    0.197803  0.074823             578
1     2  0.147322    0.206356 -0.059035            1266
2     3  0.227050    0.175719  0.051332            2760
3     4  0.273045    0.193043  0.080002             694
4     5  0.221357    0.201732  0.019625             922
OOF RMSE: 0.2328
K-Fold 교차검증(rain_int_cols 포함) 모델 앙상블(평균) Val RMSE:  0.1922
